<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Truth_Probe_Generalization_Across_Hyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

prasadmahadik_cities_path = kagglehub.dataset_download('prasadmahadik/cities')
prasadmahadik_companies_path = kagglehub.dataset_download('prasadmahadik/companies')
prasadmahadik_spanish_en_path = kagglehub.dataset_download('prasadmahadik/spanish-en')
prasadmahadik_counter_fact_path = kagglehub.dataset_download('prasadmahadik/counter-fact')
prasadmahadik_larger_than_path = kagglehub.dataset_download('prasadmahadik/larger-than')
prasadmahadik_common_claim_path = kagglehub.dataset_download('prasadmahadik/common-claim')

print('Data source import complete.')


# Setup files

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import datasets

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import os
import json
import pandas as pd
import gc

from google.colab import files
import io

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Creating Training Data for Probe

In [ ]:
statements_data = [
    {"text": "The Earth is the third planet from the Sun.", "label": "truth"},
    {"text": "Humans can breathe underwater without any equipment.", "label": "lie"},
    {"text": "Water boils at 100 degrees Celsius at sea level.", "label": "truth"},
    {"text": "The moon is made of cheese.", "label": "lie"},
    {"text": "Birds are mammals.", "label": "lie"},
    {"text": "The capital of France is Paris.", "label": "truth"},
    {"text": "A square has five equal sides.", "label": "lie"},
    {"text": "The highest mountain in the world is Mount Everest.", "label": "truth"},
    {"text": "Fish can survive out of water for several hours.", "label": "lie"},
    {"text": "The Pacific Ocean is the largest ocean on Earth.", "label": "truth"},
    {"text": "Africa is the largest continent on Earth.", "label": "truth"},
    {"text": "Gold is a non-metal.", "label": "lie"},
    {"text": "Issac Newton discovered the law of gravity", "label" : "truth"},
    {"text": "The Machinist is a movie starring Leonardo DiCaprio", "label": "lie"},
    {"text": "Gold is the most costly metal.", "label": "lie"},
    {"text": "The Pacific Ocean is the largest ocean on Earth.", "label": "truth"},
    {"text": "Bread is baked in a quarry.", "label": "lie"},
    {"text": "Electricity is due to flow of electrons.", "label": "truth"},
    {"text": "Mathematics is the study of fish stews over the world", "label": "lie"},
    {"text": "Sachin Tendulkar has the highest runs in Cricket", "label": "truth"}
]

## Downoading Model

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

## Creating the probes for all layers from Difference In Means

In [ ]:
truth_text_data = [data['text']for data in statements_data if data['label'] == 'truth']
lie_text_data = [data['text']for data in statements_data if data['label'] == 'lie']

truth_label =  [data['label']for data in statements_data if data['label'] == 'truth']
lie_label =  [data['label']for data in statements_data if data['label'] == 'truth']

truth_tokens = tokenizer(truth_text_data, padding = True)
lie_tokens = tokenizer(lie_text_data, padding = True)

truth_token_tensors = torch.tensor(truth_tokens['input_ids'], device = device)
lie_token_tensors = torch.tensor(lie_tokens['input_ids'], device = device)

truth_output = model(truth_token_tensors, output_hidden_states = True)
lie_output = model(lie_token_tensors, output_hidden_states = True)


layer_outputs_truth = torch.stack(truth_output.hidden_states, dim = 0) # [layer, batch, seq, d_model]
layer_outputs_lie = torch.stack(lie_output.hidden_states, dim = 0) # [layer, batch, seq, d_model]

In [ ]:
layer_outputs_truth_mean = layer_outputs_truth.mean(dim = (1,2))
layer_outputs_lie_mean = layer_outputs_lie.mean(dim = (1,2))

In [ ]:
probes_by_layer = layer_outputs_truth_mean - layer_outputs_lie_mean

In [ ]:
uploaded = files.upload()



In [ ]:
file_bytes = uploaded['cities (2).csv']

In [ ]:
test_db = pd.read_csv('/kaggle/input/cities/cities.csv')

In [ ]:
test_statements = list(test_db['statement'])
test_labels = list(test_db['label'])

In [ ]:
test_tokens = tokenizer(test_statements, padding = True)
test_token_tensor = torch.tensor(test_tokens['input_ids'], device = device)

with torch.no_grad():  # This prevents gradient storage
  test_activations = model(test_token_tensor, output_hidden_states = True).hidden_states

In [ ]:
test_activations_layers = torch.stack(test_activations, dim = 0)
probes_by_layer = probes_by_layer.unsqueeze(1).unsqueeze(1)

probe_scores = (test_activations_layers * probes_by_layer).sum(dim=-1)

probe_scores = probe_scores.mean(dim=-1)


In [ ]:
probe_scores_np = probe_scores.detach().cpu().float().numpy()
test_labels_np = np.array(test_labels)

for layer in range(probe_scores_np.shape[0]):
    auroc = roc_auc_score(test_labels_np, probe_scores_np[layer])
    print(f"Layer {layer}: AUROC = {auroc:.3f}")

The AUROC scores were pretty bad for all the layers - and it seems that the clear reason for this is that the truth and lie facts don't have counterfactuals for the probes to capture the difference of. Without the counterfactuals - the probe is probably capturing things that are semantic..

The solution would be -
I have the cities dataset - and it has clear counterfactual pairs - so, create general purpose function that extracts the activations, and another that gets the probes from the activations and their differences.


In [ ]:
def get_activations(model, tokenizer, texts, device, batch_size=8):
    """
    Get activations for a list of texts.

    Args:
        model: The language model
        tokenizer: The tokenizer
        texts: List of strings
        device: 'cuda' or 'cpu'
        batch_size: Number of texts to process at once

    Returns:
        torch.Tensor of shape [n_layers, n_samples, seq_len, d_model]
    """

    all_tokens = tokenizer(texts, padding='longest', return_tensors='pt')
    max_len = all_tokens['input_ids'].shape[1]

    all_activations = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc = 'Getting batched activations'):
            batch_texts = texts[i:i+batch_size]
            # Pad to the global max length
            tokens = tokenizer(batch_texts, padding='max_length', max_length=max_len, return_tensors='pt')
            token_ids = tokens['input_ids'].to(device)

            output = model(token_ids, output_hidden_states=True)
            batch_activations = torch.stack(output.hidden_states, dim=0)
            all_activations.append(batch_activations)

            del token_ids, output
            torch.cuda.empty_cache()

    # Now all batches have same seq_len, concatenation works!
    return torch.cat(all_activations, dim=1)


def compute_mean_activations(activations, labels, label_value):
    """
    Compute mean activations for samples with specific label.

    Args:
        activations: [n_layers, n_samples, seq, d_model]
        labels: List of labels corresponding to samples
        label_value: The label to filter by (e.g., 'truth', 1, etc.)

    Returns:
        torch.Tensor of shape [n_layers, 1, 1, d_model]
    """
    # Find indices matching the label
    label_indices = [i for i, label in enumerate(labels) if label == label_value]


    filtered = activations[:, label_indices, :, :]
    mean_activations = filtered.mean(dim=(1, 2), keepdim=True)

    return mean_activations


def create_difference_probe(truth_mean, lie_mean):
    """
    Create difference-of-means probe.

    Args:
        truth_mean: [n_layers, 1, 1, d_model]
        lie_mean: [n_layers, 1, 1, d_model]

    Returns:
        probe: [n_layers, 1, 1, d_model]
    """
    return truth_mean - lie_mean


def evaluate_probe(probe, test_activations, test_labels):

    from sklearn.metrics import roc_auc_score

    binary_labels = [1 if label in ['truth', 1] else 0 for label in test_labels]

    # Do multiplication AND sum on GPU in one step
    scores = (test_activations * probe).sum(dim=-1).mean(dim=-1)

    # Move small tensor to CPU
    scores_np = scores.float().cpu().numpy()

    # Clear GPU immediately
    del test_activations, probe, scores
    torch.cuda.empty_cache()

    results = {}
    for layer in range(scores_np.shape[0]):
        auroc = roc_auc_score(binary_labels, scores_np[layer])
        results[layer] = auroc

    return results

def train_probe(texts, labels, label_values, model, tokenizer):
    label_value_lie = label_values[0]
    label_value_truth = label_values[1]
    train_activations = get_activations(model, tokenizer, texts, device, batch_size=8)
    lie_mean = compute_mean_activations(train_activations, labels = labels, label_value = label_value_lie)
    truth_mean = compute_mean_activations(train_activations, labels = labels, label_value = label_value_truth)
    probe = create_difference_probe(truth_mean, lie_mean)

    return probe

def evaluate_probe_on_text(probe, test_texts, test_labels, model, tokenizer):

    test_activations = get_activations(model, tokenizer, test_texts, device, batch_size=8)

    binary_labels = [1 if label in ['truth', 1] else 0 for label in test_labels]

    # Do multiplication AND sum on GPU in one step
    scores = (test_activations * probe).sum(dim=-1).mean(dim=-1)

    # Move small tensor to CPU
    scores_np = scores.float().cpu().numpy()

    # Clear GPU immediately
    del test_activations, probe, scores
    torch.cuda.empty_cache()

    results = {}
    for layer in range(scores_np.shape[0]):
        auroc = roc_auc_score(binary_labels, scores_np[layer])
        results[layer] = auroc

    return results

def compute_probe_from_activations(activations, labels):
    """Compute probe from pre-computed activations"""
    lie_mean = compute_mean_activations(activations, labels, 0)
    truth_mean = compute_mean_activations(activations, labels, 1)
    probe = create_difference_probe(truth_mean, lie_mean)
    return probe



def evaluate_probe_on_activations(probe, activations, labels):
    """Evaluate using pre-computed activations"""
    binary_labels = [1 if label in ['truth', 1] else 0 for label in labels]

    scores = (activations * probe).sum(dim=-1).mean(dim=-1)
    scores_np = scores.float().cpu().numpy()

    results = {}
    for layer in range(scores_np.shape[0]):
        auroc = roc_auc_score(binary_labels, scores_np[layer])
        results[layer] = auroc

    return results

In [ ]:
texts = list(test_db['statement'])
labels = list(test_db['label'])
labels = list(map(lambda x: int(x), labels))

label_value_truth = 1
label_value_lie = 0
label_values = [label_value_lie, label_value_truth]

probe = train_probe(texts, labels, label_values, model, tokenizer)

In [ ]:
uploaded = files.upload()

In [ ]:
file_bytes = uploaded['companies_true_false.csv']
true_test_db = pd.read_csv(io.BytesIO(file_bytes))

In [ ]:
true_test_db = pd.read_csv('/kaggle/input/companies/companies_true_false.csv')
test_texts = list(true_test_db['statement'])
test_labels = list(true_test_db['label'])

test_activations = get_activations(model, tokenizer, test_texts, device, batch_size=8)

In [ ]:
results = evaluate_probe(probe, test_activations, test_labels)
for layer, auroc in results.items():
    print(f'Layer {layer} AUROC ---> {auroc :.2f}')

Now that we have good pipeline of training probes, evaluation and related functions, we can sweep through datasets, and find evaluation scores for different combination of probes.

So, I should take in a list of paths, and create dbs, then create texts, and create labels and label_values - this could be called get_all_data -> texts, labels, label_values
should this be a dict or a list?
I think it should be a dict - clear and no confusion.

then I should have another function that trains on db, and another that evaluates and finds its best probe. After this, things can be thrown..this can be repeated for a probe, and all



In [ ]:
def load_datasets(datasets_paths):
    """
    Args:
        dataset_paths: dict like {'cities': 'path/to/cities.csv', 'companies': ...}

    Returns:
        dict: {
            'cities': {'texts': [...], 'labels': [0,1,0,...]},
            'companies': {'texts': [...], 'labels': [0,1,1,...]},
            ...
        }
    """
    datasets_dict = {}

    for dataset_name, path in datasets_paths.items():
        db = pd.read_csv(path)
        db_text = list(db['statement'])
        db_label = list(db['label'])
        datasets_dict[dataset_name] = {'texts': db_text,
                                     'labels':db_label}
    return datasets_dict

In [ ]:
datasets_paths = {'spanish_db':'/kaggle/input/spanish-en/sp_en_trans.csv',
                'common_claim_db':'/kaggle/input/common-claim/common_claim_true_false.csv',
                'counter_fact_db' :'/kaggle/input/counter-fact/counterfact_true_false.csv',
                'larger_than_db' : '/kaggle/input/larger-than/larger_than.csv',
                'cities_db' : '/kaggle/input/cities/cities.csv',
                'companies_db' : '/kaggle/input/companies/companies_true_false.csv',
                }

datasets_dict = load_datasets(datasets_paths)

# Streaming functions for training and evaluating probes

Computing probes via diff of means was easier for smaller datasets, but for larger, it needs batching and calculating means across batches. The same goes for evaluating. Both functions are made to support streaming.


In [ ]:
def compute_probe_streaming(texts, labels, model, tokenizer, batch_size=8):
    """
    Compute probe using streaming/incremental mean calculation.
    Never stores all activations at once!
    """

    truth_sum = None
    lie_sum = None
    truth_count = 0
    lie_count = 0


    all_tokens = tokenizer(texts, padding='longest', return_tensors='pt')
    max_len = all_tokens['input_ids'].shape[1]
    del all_tokens

    # Process in batches
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Computing probe'):
            # Get batch
            batch_texts = texts[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]


            tokens = tokenizer(batch_texts, padding='max_length', max_length=max_len, return_tensors='pt')
            token_ids = tokens['input_ids'].to(device)
            output = model(token_ids, output_hidden_states=True)
            batch_activations = torch.stack(output.hidden_states, dim=0)  # [n_layers, batch, seq, d_model]

            # Split by label and accumulate
            for idx, label in enumerate(batch_labels):
                # Get activations for this sample: [n_layers, seq, d_model]
                sample_act = batch_activations[:, idx, :, :]

                if label == 1:  # truth
                    if truth_sum is None:
                        truth_sum = sample_act.sum(dim=1)  # Sum over sequence: [n_layers, d_model]
                    else:
                        truth_sum += sample_act.sum(dim=1)
                    truth_count += sample_act.shape[1]  # Add sequence length
                else:  # lie
                    if lie_sum is None:
                        lie_sum = sample_act.sum(dim=1)
                    else:
                        lie_sum += sample_act.sum(dim=1)
                    lie_count += sample_act.shape[1]

            # Clean up batch
            del token_ids, output, batch_activations
            torch.cuda.empty_cache()

    # Compute final means
    truth_mean = (truth_sum / truth_count).unsqueeze(1).unsqueeze(1)  # [n_layers, 1, 1, d_model]
    lie_mean = (lie_sum / lie_count).unsqueeze(1).unsqueeze(1)

    probe = truth_mean - lie_mean
    return probe

def evaluate_probe_streaming(probe, texts, labels, model, tokenizer, batch_size=8):
    """
    Evaluate probe using streaming - never stores all activations at once
    """

    all_tokens = tokenizer(texts, padding='longest', return_tensors='pt')
    max_len = all_tokens['input_ids'].shape[1]
    del all_tokens

    n_layers = probe.shape[0]
    all_scores = [[] for _ in range(n_layers)]
    binary_labels = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Evaluating'):
            batch_texts = texts[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]


            tokens = tokenizer(batch_texts, padding='max_length', max_length=max_len, return_tensors='pt')
            token_ids = tokens['input_ids'].to(device)
            output = model(token_ids, output_hidden_states=True)
            batch_activations = torch.stack(output.hidden_states, dim=0)

            # Compute scores for this batch
            batch_scores = (batch_activations * probe).sum(dim=-1).mean(dim=-1)  # [n_layers, batch]

            # Accumulate scores per layer
            for layer in range(n_layers):
                all_scores[layer].extend(batch_scores[layer].cpu().float().numpy().tolist())

            binary_labels.extend([1 if label in ['truth', 1] else 0 for label in batch_labels])


            del token_ids, output, batch_activations, batch_scores
            torch.cuda.empty_cache()

    # Compute AUROC for each layer
    results = {}
    for layer in range(n_layers):
        auroc = roc_auc_score(binary_labels, all_scores[layer])
        results[layer] = auroc

    return results


def train_all_probes(datasets, model, tokenizer, batch_size=8):
    """Train all probes using streaming computation"""
    probes = {}

    for dataset_name, data in datasets.items():
        print(f'\nTraining probe on {dataset_name}')


        probe = compute_probe_streaming(
            data['texts'],
            data['labels'],
            model,
            tokenizer,
            batch_size=batch_size
        )


        print('Evaluating on same dataset')
        layer_aurocs = evaluate_probe_streaming(
            probe,
            data['texts'],
            data['labels'],
            model,
            tokenizer,
            batch_size=batch_size
        )

        best_layer = max(layer_aurocs, key=layer_aurocs.get)
        print(f'Best layer: {best_layer}, AUROC: {layer_aurocs[best_layer]:.3f}')

        # Store full probe (all layers) + best layer info
        probes[dataset_name] = {
            'probe': probe,
            'best_layer': best_layer,
            'train_auroc': layer_aurocs[best_layer]
        }

        # Cleanup
        del probe, layer_aurocs
        torch.cuda.empty_cache()
        gc.collect()

    return probes

# Training Probes for each Dataset

**These are trained for each dataset, on all layers, and out of that the best layer is found based on its AUROC score.**

In [ ]:
all_probes = train_all_probes(datasets_dict, model, tokenizer, batch_size=8)

This is a function for evaluating the probe trained on one of the datasets, on all other datasets. Its output is a nice dict containing the train database name, its best layer, its train_auroc and test results across all other datasets

In [ ]:
def evaluate_probe_on_all_datasets(
    train_dataset_name,  # which probe to use
    probes,              # the full probes dict
    datasets,            # all test datasets
    model,
    tokenizer,
    batch_size=8
):
    """
    Take ONE probe (trained on train_dataset_name) and evaluate it across ALL datasets.
    Returns generalization results for this single probe.
    """

    probe_info = probes[train_dataset_name]
    probe = probe_info['probe']  # shape: [n_layers, 1, 1, d_model]


    results = {
        'train_dataset': train_dataset_name,
        'train_best_layer': probe_info['best_layer'],
        'train_auroc': probe_info['train_auroc'],
        'test_results': {}
    }


    for test_dataset_name, test_data in datasets.items():
        print(f'\nTesting {train_dataset_name} probe on {test_dataset_name}')

        layer_aurocs = evaluate_probe_streaming(probe, test_data['texts'], test_data['labels'], model, tokenizer, batch_size=8)
        best_layer = sorted(layer_aurocs.items(), key = lambda x: x[1], reverse = True)[0][0]
        best_layer_auroc = sorted(layer_aurocs.items(), key = lambda x: x[1], reverse = True)[0][1]

        print(f'Test performance on {test_dataset_name}: Best Layer - {best_layer}, auroc {best_layer_auroc: .2f}')

        results['test_results'][test_dataset_name] = {'best_layer': best_layer,
                                                      'auroc': best_layer_auroc
                                                     }


    return results

The test results are calcualated for each of the test dataset, because its good to have them separate and not run a long single function.

In [ ]:
cities_db_test = evaluate_probe_on_all_datasets('cities_db',
                                                all_probes,
                                                datasets_dict,
                                                model,
                                                tokenizer,
                                                batch_size=8
                                                )

In [ ]:
companies_db_test = evaluate_probe_on_all_datasets(
                                                    'companies_db',
                                                    all_probes,
                                                    datasets_dict,
                                                    model,
                                                    tokenizer,
                                                    batch_size=8
                                                )

In [ ]:
larger_than_db_test = evaluate_probe_on_all_datasets(
                                                    'larger_than_db',
                                                    all_probes,
                                                    datasets_dict,
                                                    model,
                                                    tokenizer,
                                                    batch_size=8
                                                )

In [ ]:
spanish_db_test = evaluate_probe_on_all_datasets(
                                                    'spanish_db',
                                                    all_probes,
                                                    datasets_dict,
                                                    model,
                                                    tokenizer,
                                                    batch_size=8
                                                )

In [ ]:
common_claim_db_test = evaluate_probe_on_all_datasets(
                                                    'common_claim_db',
                                                    all_probes,
                                                    datasets_dict,
                                                    model,
                                                    tokenizer,
                                                    batch_size=8
                                                )

In [ ]:
counter_fact_db_test = evaluate_probe_on_all_datasets(
                                                    'counter_fact_db',
                                                    all_probes,
                                                    datasets_dict,
                                                    model,
                                                    tokenizer,
                                                    batch_size=8
                                                )

# Storing all results in a single dictionary

**The test results for all dataset and their generalization are stored in a single dict.**

In [ ]:
results = {
    'larger_than_db': larger_than_db_test,
    'cities_db': cities_db_test,
    'companies_db': companies_db_test,
    'spanish_db': spanish_db_test,
    'common_claim_db': common_claim_db_test,
    'counter_fact_db': counter_fact_db_test
}

# Plotting results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def plot_generalization_heatmap(results, figsize=(10, 8)):
    """
    1. Generalization Matrix/Heatmap
    Rows = train dataset, Cols = test dataset
    """
    dataset_names = list(results.keys())
    n = len(dataset_names)


    matrix = np.zeros((n, n))
    for i, train_name in enumerate(dataset_names):
        for j, test_name in enumerate(dataset_names):
            matrix[i, j] = results[train_name]['test_results'][test_name]['auroc']


    clean_names = [name.replace('_db', '') for name in dataset_names]


    plt.figure(figsize=figsize)
    sns.heatmap(matrix, annot=True, fmt='.2f',
                xticklabels=clean_names,
                yticklabels=clean_names,
                cmap='RdYlGn', vmin=0.5, vmax=1.0,
                cbar_kws={'label': 'AUROC'})
    plt.xlabel('Test Dataset', fontsize=12)
    plt.ylabel('Train Dataset', fontsize=12)
    plt.title('Probe Generalization Matrix\n(Diagonal = Self-Performance)', fontsize=14, pad=20)
    plt.tight_layout()
    return matrix


def plot_generalization_scores(results, figsize=(10, 6)):
    """
    2. Probe Generalization Ranking
    Average AUROC on OTHER datasets (excluding self)
    """
    gen_scores = {}
    self_scores = {}

    for train_name, result in results.items():

        self_scores[train_name] = result['train_auroc']

        # Average on other datasets
        other_aurocs = [v['auroc'] for k, v in result['test_results'].items()
                        if k != train_name]
        gen_scores[train_name] = np.mean(other_aurocs)


    clean_names = [name.replace('_db', '') for name in gen_scores.keys()]


    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(clean_names))
    width = 0.35

    bars1 = ax.bar(x - width/2, [self_scores[k] for k in gen_scores.keys()],
                   width, label='Self-Performance', alpha=0.8)
    bars2 = ax.bar(x + width/2, list(gen_scores.values()),
                   width, label='Avg on Others', alpha=0.8)

    ax.set_xlabel('Probe (Trained on)', fontsize=12)
    ax.set_ylabel('AUROC', fontsize=12)
    ax.set_title('Probe Self-Performance vs Generalization', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(clean_names, rotation=45, ha='right')
    ax.legend()
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim([0.45, 1.0])


    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    return gen_scores


def plot_layer_consistency(results, figsize=(12, 6)):
    """
    3. Layer Consistency Analysis
    Which layers work best for each probe-dataset pair?
    """
    data = []

    for train_name, result in results.items():
        for test_name, test_result in result['test_results'].items():
            data.append({
                'probe': train_name.replace('_db', ''),
                'test_dataset': test_name.replace('_db', ''),
                'best_layer': test_result['best_layer'],
                'auroc': test_result['auroc']
            })

    df = pd.DataFrame(data)

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Left: Heatmap of best layers
    pivot_layers = df.pivot(index='probe', columns='test_dataset', values='best_layer')
    sns.heatmap(pivot_layers, annot=True, fmt='.0f', cmap='viridis',
                ax=ax1, cbar_kws={'label': 'Best Layer'})
    ax1.set_title('Best Layer for Each Probe-Dataset Pair')
    ax1.set_xlabel('Test Dataset')
    ax1.set_ylabel('Probe')

    # Right: Layer distribution
    ax2.hist(df['best_layer'], bins=range(0, 33), alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Layer Number', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Distribution of Best Layers')
    ax2.axvline(x=df['best_layer'].median(), color='red',
                linestyle='--', label=f'Median: {df["best_layer"].median():.0f}')
    ax2.legend()

    plt.tight_layout()
    return df


def plot_transfer_asymmetry(results, figsize=(10, 6)):
    """
    5. Asymmetric Transfer Analysis
    Compare A→B vs B→A transfer
    """
    dataset_names = list(results.keys())
    asymmetries = []

    for i, train_a in enumerate(dataset_names):
        for j, train_b in enumerate(dataset_names):
            if i < j:

                a_to_b = results[train_a]['test_results'][train_b]['auroc']

                b_to_a = results[train_b]['test_results'][train_a]['auroc']

                asymmetry = abs(a_to_b - b_to_a)
                asymmetries.append({
                    'pair': f"{train_a.replace('_db', '')} ↔ {train_b.replace('_db', '')}",
                    'forward': a_to_b,
                    'backward': b_to_a,
                    'asymmetry': asymmetry
                })

    df = pd.DataFrame(asymmetries).sort_values('asymmetry', ascending=False)

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Left: Asymmetry magnitude
    ax1.barh(range(len(df)), df['asymmetry'], alpha=0.7)
    ax1.set_yticks(range(len(df)))
    ax1.set_yticklabels(df['pair'])
    ax1.set_xlabel('|AUROC_forward - AUROC_backward|', fontsize=11)
    ax1.set_title('Transfer Asymmetry by Dataset Pair')
    ax1.invert_yaxis()

    # Right: Forward vs Backward scatter
    ax2.scatter(df['forward'], df['backward'], s=100, alpha=0.6)
    ax2.plot([0.5, 1], [0.5, 1], 'r--', alpha=0.5, label='Perfect symmetry')
    for idx, row in df.iterrows():
        ax2.annotate(row['pair'], (row['forward'], row['backward']),
                    fontsize=8, alpha=0.7)
    ax2.set_xlabel('Forward Transfer AUROC', fontsize=11)
    ax2.set_ylabel('Backward Transfer AUROC', fontsize=11)
    ax2.set_title('Transfer Symmetry')
    ax2.legend()
    ax2.set_xlim([0.5, 1.0])
    ax2.set_ylim([0.5, 1.0])

    plt.tight_layout()
    return df


def create_full_report(results, save_path=None):
    """
    Generate all visualizations at once
    """
    print("Generating full probe analysis report...")

    print("\n1. Generalization Heatmap")
    matrix = plot_generalization_heatmap(results)
    plt.show()

    print("\n2. Generalization Scores")
    gen_scores = plot_generalization_scores(results)
    plt.show()
    print("\nGeneralization Scores:")
    for name, score in sorted(gen_scores.items(), key=lambda x: x[1], reverse=True):
        print(f"  {name}: {score:.3f}")

    print("\n3. Layer Consistency")
    layer_df = plot_layer_consistency(results)
    plt.show()

    print("\n4. Transfer Asymmetry")
    asym_df = plot_transfer_asymmetry(results)
    plt.show()
    print("\nMost Asymmetric Pairs:")
    print(asym_df.head())

    if save_path:
        print(f"\nSaving report to {save_path}")
        # Save summary stats
        summary = {
            'generalization_scores': gen_scores,
            'layer_stats': layer_df.groupby('probe')['best_layer'].describe(),
            'asymmetries': asym_df
        }
        # You can pickle or save as JSON here

    return {
        'gen_scores': gen_scores,
        'layer_df': layer_df,
        'asymmetry_df': asym_df,
        'matrix': matrix
    }


In [ ]:
report = create_full_report(results)